<a href="https://colab.research.google.com/github/niharnandala/ntsb-safety-review-agent/blob/main/notebooks/01_data_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Data checks

Before we train anything, we need to know if our answer keys are actually right.
The answer key for each practice question is NTSB's grade for an organisation's reply.
If those grades are wrong, the model learns wrong stuff and we'd never know why.

**Data:** NTSB CAROL export — 10,000 recommendations, ~102k letters between NTSB and the organisations.
**Runtime:** CPU is enough here, no GPU needed (Runtime → Change runtime type → CPU).

## Load the data
Upload the CAROL zip, unzip it, and load the JSON.
Each record = one recommendation → its recipients → all the letters back and forth, in one list.

In [1]:
from google.colab import files
files.upload()   # a "Choose files" button shows up -> pick the CAROL zip

!unzip -o -q "carol_recommendations_data (1).zip" -d data
!ls -lh data

Saving carol_recommendations_data (1).zip to carol_recommendations_data (1).zip
total 198M
-rw-r--r-- 1 root root 198M Nov 10 11:19 'carol_recommendations_data (1).json'


In [2]:
import json, glob, re, random
from collections import Counter

path = glob.glob("data/*.json")[0]
with open(path) as f:
    recs = json.load(f)

print(len(recs), "recommendations")

10000 recommendations


## Pair each reply with NTSB's answer
For every reply from an organisation, we take the letter right after it.
If that letter is from NTSB, we treat it as NTSB's answer to that reply.

Two small fixes while doing this:
- Some letters are marked as "from NTSB" but actually start with "-From <someone at the org>". Those are really org letters.
- Empty letters are skipped.

The grade itself isn't a field in the data. It's only written inside NTSB's letter, in quotes,
like *classified "Closed—Acceptable Action."* So we pull it out with a regex.

In [3]:
# NTSB writes the grade in quotes, e.g. classified "Open--Acceptable Response."
GRADE = re.compile(r'["“]\s*((?:Open|Closed)\s*[-–—]+\s*[A-Za-z /]+?)\s*[."”,]', re.I)

def text(letter):
    return (letter.get("ResponseSummary") or "").strip()

def from_ntsb(letter):
    # some letters are flagged as NTSB but really start with "-From <org person>"
    return bool(letter.get("IsFromNtsb")) and not text(letter).startswith("-From")

def letters_in_order(addressee):
    # skip empty letters, sort by date
    letters = [l for l in (addressee.get("Correspondence") or []) if text(l)]
    return sorted(letters, key=lambda l: l.get("CorrespondenceDate", ""))

# round 1: reply -> next letter if it's from NTSB, take the LAST quoted grade in it
pairs = []
for r in recs:
    for a in r["Addressees"]:
        letters = letters_in_order(a)
        for i, l in enumerate(letters):
            if from_ntsb(l):
                continue
            nxt = letters[i + 1] if i + 1 < len(letters) else None
            if nxt is None or not from_ntsb(nxt):
                continue   # no NTSB answer right after this reply
            found = GRADE.findall(text(nxt))
            pairs.append({
                "rec": r["Srid"],
                "reply": text(l),
                "ntsb": text(nxt),
                "grade": found[-1] if found else None,
            })

with_grade = [p for p in pairs if p["grade"]]
print(len(pairs), "reply → NTSB pairs")
print(len(with_grade), "of them have a readable grade")

27828 reply → NTSB pairs
12738 of them have a readable grade


## Check 1 — are the grades we read actually correct?
Looked at 10 random ones by eye.

**Result:** 7 correct, 2 weak, 1 wrong.
The wrong one was M-11-023: the letter graded M-10-2 and we took that grade.
That's the first hint that one NTSB letter can talk about several recommendations.

In [4]:
random.seed(1)
for p in random.sample(with_grade, 10):
    pos = p["ntsb"].find(p["grade"])
    print("REC:", p["rec"], "| GRADE READ:", p["grade"])
    print("CONTEXT:", p["ntsb"][max(0, pos - 200): pos + 60])
    print("-" * 80)

REC: H-95-015 | GRADE READ: CLOSED--ACCEPTABLE ACTION
CONTEXT: DE CROSSINGS THAT HAVE PREEMPTION CURRENTLY IN PLACE & THAT NONE OF THESE CROSSINGS WERE FOUND TO PRESENT A SAFETY PROBLEM.  BASED ON THE INFO CONTAINED IN YOUR RESPONSE, H-95-15 HAS BEEN CLASSIFIED "CLOSED--ACCEPTABLE ACTION."
--------------------------------------------------------------------------------
REC: M-01-010 | GRADE READ: Closed-Unacceptable Action
CONTEXT: eing kept informed of your progress on this initiative and notified when work has been completed fleetwide, at which time we will consider reclassifying the recommendation from its current status of "Closed-Unacceptable Action."
--------------------------------------------------------------------------------
REC: A-86-011 | GRADE READ: Closed--Acceptable Action
CONTEXT: at
Washington National Airport and for your assurance that it is in full compliance
with all directives pertaining to ATC Specialist Proficiency Requirements. This
recommendation is classifie

## Check 2 — can we use the final status as a label?
The idea was to also use each thread's final status as the answer for the last reply.
That only works if NTSB actually wrote a letter after that last reply.

**Result:** 13,122 of 14,723 threads have one. So about 1 in 9 would get a wrong label.
**Decision:** drop final-status labels completely. Only use grades NTSB actually wrote.

In [5]:
threads = 0
answered = 0
for r in recs:
    for a in r["Addressees"]:
        letters = letters_in_order(a)
        reply_spots = [i for i, l in enumerate(letters) if not from_ntsb(l)]
        if not reply_spots:
            continue
        threads += 1
        after_last = letters[reply_spots[-1] + 1:]
        if any(from_ntsb(l) for l in after_last):
            answered += 1

print(threads, "threads with at least one reply")
print(answered, "have an NTSB letter after the last reply")

14723 threads with at least one reply
13122 have an NTSB letter after the last reply


## Check 3 — enough examples per grade?
NTSB uses a lot of different wordings, so we squeeze them into a few classes.

**Result:** most classes have hundreds to thousands of examples. "Exceeds" only has 34.
**Decision:** merge Exceeds into Closed – Acceptable → 7 classes.
Open – Acceptable is ~40% of everything, so later we'll use balanced sampling and score each class separately.

In [6]:
# first version of the classes (8, Exceeds still separate)
def to_class(grade):
    g = grade.lower()
    side = "Open" if g.startswith("open") else "Closed"
    if "await" in g:        return None   # "await response" isn't a grade
    if "exceed" in g:       return "Closed – Exceeds"
    if "unacceptable" in g: return side + " – Unacceptable"
    if "alternate" in g:    return side + " – Acceptable Alternate"
    if "acceptable" in g:   return side + " – Acceptable"
    return "Closed – Other" if side == "Closed" else None

counts = Counter(to_class(p["grade"]) for p in with_grade)
for name, n in counts.most_common():
    print(f"{n:6}  {name}")

  4956  Open – Acceptable
  3904  Closed – Acceptable
  1171  Open – Unacceptable
   771  Closed – Unacceptable
   635  Closed – Acceptable Alternate
   540  Closed – Other
   412  None
   315  Open – Acceptable Alternate
    34  Closed – Exceeds


## Why the M-11-023 label was wrong
NTSB often writes one letter covering many recommendations, and the database puts a copy of that same
letter into every case it mentions. Example: a Coast Guard semiannual update covering 42 recommendations.
NTSB's answer had 4 parts, and only Part 1 actually gave new grades (M-09-15, M-09-16, M-10-2).
M-11-023 was just listed with its old status from 2012.

So for every case, we have to find the part of the letter that talks about *our* case.

**Numbers:** 27,828 NTSB answers total. 949 have more than one quoted grade (the risky ones).
4,091 mention more than one recommendation ID, but most of those are just references, not grades.

In [7]:
# any recommendation ID: mode letter + year + number, e.g. M-10-2, A-91-26
REC_ID = re.compile(r"\b[AHIMPR]\s*-?\s*\d{2}\s*-\s*\d{1,3}\b")

multi_grade = [p for p in pairs if len(GRADE.findall(p["ntsb"])) > 1]
multi_rec   = [p for p in pairs if len(set(REC_ID.findall(p["ntsb"]))) > 1]

print(len(pairs), "NTSB answers in total")
print(len(multi_grade), "contain more than one grade")
print(len(multi_rec), "mention more than one recommendation ID")

# every grade in the M-11-023 letter, with the text just before it
ex = next((p for p in multi_rec if p["rec"] == "M-11-023"), multi_rec[0])
print("\nEXAMPLE — pair is about", ex["rec"])
for m in GRADE.finditer(ex["ntsb"]):
    print("…", ex["ntsb"][max(0, m.start() - 150): m.end()].replace("\n", " "))
    print()

27828 NTSB answers in total
949 contain more than one grade
4091 mention more than one recommendation ID

EXAMPLE — pair is about M-11-023
… SA) as they relate to determining merchant mariner fitness for duty. Pending completion of these efforts, Safety Recommendation M-09-15 is classified “Open—Acceptable Alternate Response”

… on of these efforts, Safety Recommendation M-09-15 is classified “Open—Acceptable Alternate Response” and Safety Recommendation M 09-16 is classified “Open—Acceptable Response.

…  case-by-case basis and only when operational safety is not compromised. Because this action satisfies Safety Recommendation M 10-2, it is classified “Closed—Acceptable Action.



In [8]:
# the full reply + full NTSB answer for that case
ex = [p for p in pairs if p["rec"] == "M-11-023" and "M 10-2" in p["ntsb"]]
print(len(ex), "matching pair(s)\n")

for p in ex:
    print("=" * 30, "ORGANISATION'S REPLY", "=" * 30)
    print(p["reply"])
    print("\n" + "=" * 30, "NTSB'S FULL ANSWER", "=" * 30)
    print(p["ntsb"])

1 matching pair(s)

============================== ORGANISATION'S REPLY ==============================
-From Peter V. Neffenger, Vice Admiral, U.S. Coast Guard, Deputy Commandant for Operations: Please find enclosed our agreed upon semiannual update of actions on safety recommendations issued to the Coast Guard by the National Transportation Safety Board that are currently assigned an "open" status by the Board and are awaiting Coast Guard response.

There are currently 42 safety recommendations with an "open" status issued to the Coast Guard. Of those, we attest that our actions are complete for six, six are pending resolution, and five require long-term agency action. Updates for the remaining 25 have been or will be provided in separate correspondence. Enclosure (1) provides specific information for each recommendation.

Initial responses and updates of action since the previous update for the following 25 recommendations have been or are being provided (recommendations M-12-8, M-12

## Round 2 — tie the grade to our recommendation's ID
For each grade in the letter, look at the text before it. If our ID shows up there, the grade is ours.
If no ID shows up, accept the grade only when the letter has exactly one grade. Otherwise drop the pair.
Also: "Superseded" goes to Closed – Other, and Exceeds is merged into Closed – Acceptable.

**Result (20 samples):** 17 correct, 1 lucky (NTSB typo'd the ID), 2 wrong. Two new problems:
- **Intro lists:** many letters start by listing every recommendation they cover, so the first grade
  "saw" our ID in the intro and we wrongly claimed it.
- **Old recaps:** letters often say *"in its 2005 letter, the Board classified X as …"* before giving
  the new grade. We grabbed the old one.

In [9]:
def id_parts(srid):
    mode, yy, num = srid.split("-")[:3]
    return mode, yy, int(re.sub(r"\D", "", num) or 0)

def mentions_rec(passage, srid):
    mode, yy, num = id_parts(srid)
    # single ID, e.g. "A-91-26", "A-91-026", "A 91-26"
    single = re.compile(rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*{num}\b", re.I)
    if single.search(passage):
        return True
    # ranges, e.g. "H-93-5 through -7"
    rng = re.compile(rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*(\d+)\s*(?:through|thru|to)\s*-?\s*(?:{mode}\s*-?\s*{yy}\s*-\s*)?0*(\d+)", re.I)
    return any(int(lo) <= num <= int(hi) for lo, hi in rng.findall(passage))

def grade_for_rec(ntsb_text, srid):
    matches = list(GRADE.finditer(ntsb_text))
    if not matches:
        return None
    start = 0
    for m in matches:
        passage = ntsb_text[start:m.end()]   # text since the previous grade, up to this one
        if mentions_rec(passage, srid):
            return m.group(1)
        start = m.end()
    if len(matches) == 1:                     # no ID mentioned, but only one grade in the letter
        return matches[0].group(1)
    return None                               # several grades, none clearly ours -> drop

In [10]:
# the 7 classes we keep
def to_class(grade):
    g = grade.lower()
    side = "Open" if g.startswith("open") else "Closed"
    if "await" in g:
        return None
    if "supersed" in g or "superced" in g:      # NTSB spells it both ways
        return "Closed – Other"
    if "unacceptable" in g:
        return side + " – Unacceptable"
    if "alternate" in g:
        return side + " – Acceptable Alternate"
    if "acceptable" in g or "exceed" in g:      # Exceeds merged in (only 34 examples)
        return side + " – Acceptable"
    return "Closed – Other" if side == "Closed" else None

In [11]:
pairs = []
for r in recs:
    for a in r["Addressees"]:
        letters = letters_in_order(a)
        for i, l in enumerate(letters):
            if from_ntsb(l) or i + 1 >= len(letters) or not from_ntsb(letters[i + 1]):
                continue
            nxt = letters[i + 1]
            grade = grade_for_rec(text(nxt), r["Srid"])
            label = to_class(grade) if grade else None
            pairs.append({"rec": r["Srid"], "reply": text(l), "ntsb": text(nxt), "grade": grade, "label": label})

labelled = [p for p in pairs if p["label"]]
print(len(pairs), "pairs →", len(labelled), "with a trusted label")

for name, n in Counter(p["label"] for p in labelled).most_common():
    print(f"{n:6}  {name}")

27828 pairs → 12210 with a trusted label
  4973  Open – Acceptable
  3967  Closed – Acceptable
  1130  Open – Unacceptable
   678  Closed – Unacceptable
   618  Closed – Acceptable Alternate
   536  Closed – Other
   308  Open – Acceptable Alternate


In [12]:
random.seed(7)
for p in random.sample(labelled, 20):
    pos = p["ntsb"].find(p["grade"])
    print("REC:", p["rec"], "| LABEL:", p["label"])
    print("CONTEXT:", p["ntsb"][max(0, pos - 250): pos + 60].replace("\n", " "))
    print("-" * 80)

REC: A-86-051 | LABEL: Open – Acceptable
 remains as "Open--Acceptable Action."
--------------------------------------------------------------------------------
REC: A-95-112 | LABEL: Closed – Unacceptable
CONTEXT: E ONE-ENGINE INOPERATIVE TAKEOFF PROCEDURES OR PROVIDE PERFORMANCE FIGURES & RUNWAY REQUIREMENTS COMPATIBLE WITH THE ACHIEVEMENT OF MAXIMUM THRUST AT AN APPROPRIATE SPEED GREATER THAN GROUND MINIMUM CONTROL SPEED.  THEREFORE, A-95-112 IS CLASSIFIED "CLOSED--UNACCEPTABLE ACTION."
--------------------------------------------------------------------------------
REC: M-90-085 | LABEL: Open – Acceptable
CONTEXT:  its findings to the Coast Guard. The Safety Board would appreciate receiving a response from the Coast Guard on its intended actions after it has reviewed the SNAME report. Pending your further reply, Safety Recommendation M-90-85 is classified as "Open--Acceptable Response."
--------------------------------------------------------------------------------
REC: A-94-197 |

## Round 3 — tighter rules
- Only look at the grade's own sentence + the one before it (fixes the intro-list problem).
- Skip "old news" sentences like *"in its … letter"*, *"previously classified"* (fixes recaps).
- If our case gets graded more than once, the last grade wins.
- Handle lists like *"H-04-8, -9, and -14"*.
- One grade but no ID? Only accept it if the letter doesn't name any other recommendation.
- Drop the pair if the org's reply is clearly about other cases.
- Tag batch letters so we can score them separately later.

**Result:** 11,168 trusted pairs kept. Dropped 16,267 with no clear grade for our case
(mostly letters with no grade in them at all) and 393 where the reply wasn't about our case.
2,539 of the kept pairs are from batch letters, so we didn't throw batch letters away completely.

| Class | Count |
|---|---|
| Open – Acceptable | 4,538 |
| Closed – Acceptable | 3,589 |
| Open – Unacceptable | 999 |
| Closed – Unacceptable | 645 |
| Closed – Acceptable Alternate | 591 |
| Closed – Other | 521 |
| Open – Acceptable Alternate | 285 |

In [13]:
ANY_ID = re.compile(r"\b[AHIMPR]\s*-?\s*\d{2}\s*-\s*\d{1,3}\b")
# "old news" phrases: grades in these sentences are recaps, not new decisions
RECAP = re.compile(
    r"\b(in (its|our|the board's)\b[^.]{0,60}\bletter|previously classified|had (been )?classified|was classified|were classified)",
    re.I,
)

def mentions_rec(passage, srid):
    mode, yy, num = id_parts(srid)
    # single ID: "A-91-26", "A-91-026", "A 91-26"
    if re.search(rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*{num}\b", passage, re.I):
        return True
    # ranges: "H-93-5 through -7"
    rng = rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*(\d+)\s*(?:through|thru|to)\s*-?\s*(?:{mode}\s*-?\s*{yy}\s*-\s*)?0*(\d+)"
    if any(int(lo) <= num <= int(hi) for lo, hi in re.findall(rng, passage, re.I)):
        return True
    # lists: "H-04-8, -9, and -14"
    lst = rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*\d+((?:\s*(?:,|and|&)\s*(?:and\s*)?-\s*0*\d+)+)"
    for tail in re.findall(lst, passage, re.I):
        if num in [int(n) for n in re.findall(r"-\s*0*(\d+)", tail)]:
            return True
    return False

def sentence_window(txt, pos):
    """The sentence that contains position `pos`, plus the sentence before it."""
    ends = [m.end() for m in re.finditer(r"[.!?][\"”']?\s+", txt[:pos])]
    start = ends[-2] if len(ends) >= 2 else 0
    return txt[start:pos]

def other_ids(txt, srid):
    # recommendation IDs in the text that aren't ours
    return {i for i in ANY_ID.findall(txt) if not mentions_rec(i, srid)}

def grade_for_rec(ntsb_text, srid):
    fresh = [m for m in GRADE.finditer(ntsb_text)
             if not RECAP.search(sentence_window(ntsb_text, m.start()))]
    ours = [m for m in fresh if mentions_rec(sentence_window(ntsb_text, m.start()), srid)]
    if ours:
        return ours[-1].group(1)              # latest grade for our case wins
    if len(fresh) == 1 and not other_ids(ntsb_text, srid):
        return fresh[0].group(1)              # one grade, and the letter is only about us
    return None                               # unclear -> drop

def reply_is_about(reply, srid):
    return mentions_rec(reply, srid) or not other_ids(reply, srid)

In [14]:
pairs = []
dropped_no_grade = dropped_reply = 0

for r in recs:
    for a in r["Addressees"]:
        letters = letters_in_order(a)
        for i, l in enumerate(letters):
            if from_ntsb(l) or i + 1 >= len(letters) or not from_ntsb(letters[i + 1]):
                continue
            reply, ntsb = text(l), text(letters[i + 1])
            grade = grade_for_rec(ntsb, r["Srid"])
            label = to_class(grade) if grade else None
            if not label:
                dropped_no_grade += 1
                continue
            if not reply_is_about(reply, r["Srid"]):
                dropped_reply += 1
                continue
            batch = bool(other_ids(ntsb, r["Srid"]) or other_ids(reply, r["Srid"]))
            pairs.append({"rec": r["Srid"], "reply": reply, "ntsb": ntsb,
                          "grade": grade, "label": label, "batch": batch})

print(len(pairs), "trusted pairs kept")
print(dropped_no_grade, "dropped: no clear grade for our case")
print(dropped_reply, "dropped: reply not about our case")
print(sum(p["batch"] for p in pairs), "of the kept pairs come from batch letters")
print()
for name, n in Counter(p["label"] for p in pairs).most_common():
    print(f"{n:6}  {name}")

11168 trusted pairs kept
16267 dropped: no clear grade for our case
393 dropped: reply not about our case
2539 of the kept pairs come from batch letters

  4538  Open – Acceptable
  3589  Closed – Acceptable
   999  Open – Unacceptable
   645  Closed – Unacceptable
   591  Closed – Acceptable Alternate
   521  Closed – Other
   285  Open – Acceptable Alternate


In [15]:
# eyeball 20 from round 3
random.seed(11)
for p in random.sample(pairs, 20):
    pos = p["ntsb"].rfind(p["grade"])
    print("REC:", p["rec"], "| LABEL:", p["label"], "| BATCH:", p["batch"])
    print("CONTEXT:", sentence_window(p["ntsb"], pos).replace("\n", " ")[-300:], "→", p["grade"])
    print("-" * 80)

REC: M-93-001 | LABEL: Closed – Acceptable | BATCH: False
CONTEXT: Although the education course is not mandatory for persons 18 years of age or older, it does require knowledge of rules to pass the test and should help to increase recreational boating safety in Illinois.  Accordingly, Safety Recommendation M-93-1 is classified " → Closed--Acceptable Action
--------------------------------------------------------------------------------
REC: H-01-039 | LABEL: Closed – Other | BATCH: False
CONTEXT: on systems, may be installed within four feet of the driver's seat back in its rearmost upright position."  These actions by manufacturers and the States work together to eliminate the safety hazard that was addressed by this recommendation; accordingly, Safety Recommendation H-01-39 is classified " → Closed--Reconsidered
--------------------------------------------------------------------------------
REC: A-93-090 | LABEL: Closed – Acceptable Alternate | BATCH: False
CONTEXT: THE SAFETY BOAR

## Where we landed
| Decision | Why |
|---|---|
| Use only grades NTSB wrote in its letters | Final status was wrong for ~1 in 9 threads |
| Find the grade in the sentence that names our case | One letter often covers many cases |
| Skip recap sentences, last grade wins | Letters mention old grades before the new one |
| 7 classes (Exceeds merged) | Only 34 Exceeds examples |
| Keep batch letters when the rules are sure, score them separately | Dropping them all would bias the data toward simple letters |

**Still to do:** check 20 samples from round 3 by eye before we build the practice cases.
**Known limit:** some batch letters still get dropped. If there's time, we'll split batch letters into
per-recommendation sections, or let a bigger model label the leftovers with a human spot-check.